# Genome Assembly

Split from `01_genome_assembly_and_advanced_ngs.ipynb` to keep this topic self-contained.

**Navigation:** [Topic overview](./01_genome_assembly_and_advanced_ngs.ipynb) · [Next: Advanced Ngs](./02_advanced_ngs.ipynb)


---

[< Previous: Numerical Methods for Bioinformatics](../16_Numerical_Methods_for_Bioinformatics/01_numerical_methods_for_bioinformatics.ipynb) | [Tier 3: Applied Bioinformatics Overview](../README.md) | [Next: Genome Assembly + Advanced Ngs: Overview >](01_genome_assembly_and_advanced_ngs.ipynb)

---

# Genome Assembly and Advanced NGS

## Tier 3 - Applied Bioinformatics

This notebook builds directly on **3.01 NGS Fundamentals** (sequencing platforms, FASTQ, basic QC, SAM/BAM, BWA/HISAT2 alignment). Here we go deeper: assembling genomes *de novo* when no reference exists, understanding the algorithms that make alignment and assembly work, assessing assembly quality, and working with long-read technologies.

### Learning Objectives
- Understand when and why *de novo* assembly is used instead of reference mapping
- Implement a toy Overlap-Layout-Consensus (OLC) assembler
- Implement a toy de Bruijn graph assembler with k-mer decomposition
- Calculate N50, L50, and NG50 assembly quality metrics from scratch
- Understand BUSCO, QUAST, and coverage-based quality assessment
- Explain scaffolding: paired-end, mate-pair, Hi-C
- Implement Burrows-Wheeler Transform and understand FM-index backward search
- Understand k-mer frequency analysis for genome size estimation (GenomeScope)
- Compare Oxford Nanopore and PacBio HiFi long-read technologies
- Design hybrid assembly strategies

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import defaultdict, Counter
import random
import math
import itertools

random.seed(42)
np.random.seed(42)

print("Libraries loaded.")

---

## Part 1: Genome Assembly Concepts

### 1.1 Reference Mapping vs. De Novo Assembly

After sequencing, you have millions of short reads (100–300 bp for Illumina). Two fundamentally different analysis routes exist:

| Approach | When to use | Key requirement |
|----------|------------|----------------|
| **Reference mapping** | Resequencing a known species | Reference genome |
| **De novo assembly** | New organism, no reference | Sufficient coverage (~50–100×) |

**De novo assembly** is needed for:
- Newly discovered organisms (environmental metagenomics, rare species)
- Highly diverged strains where mapping fails
- Structural variant discovery (inversions, translocations)
- Telomere-to-telomere "complete" reference genomes

**The shotgun assembly problem** (illustrated by the lecture's newspaper analogy):
- Many copies of the genome are shredded into fragments 200–200,000 bp long
- Each fragment is sequenced from one or both ends (100–1000 bp reads)
- No position information is retained — reads arrive in random order
- **Goal**: reconstruct the original sequence from overlapping fragments

The challenge grows exponentially with genome size and repeat content. For a bacterial genome (~5 Mb) there may be 2^200 possible reconstructions; for a mammalian genome the number is astronomical.

### 1.2 The Overlap-Layout-Consensus (OLC) Approach

OLC is the classical approach, originally used for Sanger reads and now the basis of long-read assemblers (Canu, Flye):

1. **Overlap**: Find all pairs of reads that share a suffix-prefix overlap
2. **Layout**: Build an overlap graph; reads are nodes, edges weighted by overlap length. Find a Hamiltonian path (visits each node once)
3. **Consensus**: Derive a consensus sequence by majority-vote over aligned reads

**Why it works for long reads**: overlaps are long and unique enough to resolve most repeats. **Why it fails for short reads**: O(n²) pairwise comparisons are computationally prohibitive for billions of short reads.

In [ ]:
def find_overlaps(reads: list[str], min_overlap: int = 4) -> list[tuple]:
    """Find all suffix-prefix overlaps between reads."""
    overlaps = []
    for i, r1 in enumerate(reads):
        for j, r2 in enumerate(reads):
            if i == j:
                continue
            # Find longest suffix of r1 that is a prefix of r2
            max_overlap = min(len(r1), len(r2))
            for k in range(max_overlap, min_overlap - 1, -1):
                if r1[-k:] == r2[:k]:
                    overlaps.append((i, j, k))
                    break
    return overlaps


def greedy_assemble_olc(reads: list[str], min_overlap: int = 4) -> str:
    """Greedy OLC assembler: repeatedly merge the pair with the longest overlap."""
    contigs = list(reads)
    while True:
        overlaps = find_overlaps(contigs, min_overlap)
        if not overlaps:
            break
        # Take the best overlap
        best = max(overlaps, key=lambda x: x[2])
        i, j, ov = best
        merged = contigs[i] + contigs[j][ov:]
        # Remove merged reads and add new contig
        new_contigs = [c for idx, c in enumerate(contigs) if idx not in (i, j)]
        new_contigs.append(merged)
        contigs = new_contigs
    return contigs


# Demonstration with the lecture example
reads = [
    "AGCTACAGTATGCT",
    "TACAGTATGCTTAT",
    "GTATGCTTATCTGA",
    "TGCTTATCTGATAC",
    "TGATACCTTAGCCA",
]

print("Reads:")
for r in reads:
    print(f"  {r}")

overlaps = find_overlaps(reads, min_overlap=5)
print(f"\nOverlaps found (min_overlap=5):")
print(f"{'Read i':>8} {'Read j':>8} {'Overlap':>10} {'Seq':>20}")
for i, j, ov in sorted(overlaps, key=lambda x: -x[2]):
    print(f"{i:>8} {j:>8} {ov:>10}  {reads[i][-ov:]}")

result = greedy_assemble_olc(reads, min_overlap=5)
print(f"\nAssembly result: {result[0]}")
print(f"Expected (lecture):  AGCTACAGTATGCTATCTGATACCTTAGCCA")

### 1.3 De Bruijn Graph Assembly for Short Reads

For Illumina reads (billions of 150 bp reads), OLC is too slow. The de Bruijn graph approach solves this by reducing the comparison problem:

**Key idea**: instead of comparing whole reads, decompose each read into overlapping k-mers.

For a read of length L with k-mer size k, we extract **L − k + 1** k-mers, each shifting by one position.

**De Bruijn graph construction**:
- **Nodes**: (k−1)-mers (the "left" and "right" ends of each k-mer)
- **Edges**: k-mers (an edge from the left (k−1)-mer to the right (k−1)-mer)
- Assembly = finding an **Eulerian path** (traverses each *edge* once)

This reduces Hamiltonian path (NP-hard) to Eulerian path (solvable in O(E) by Hierholzer's algorithm).

**Why k matters** (from lecture slides):
- Small k (e.g., k=2): many nodes with high in/out-degree → ambiguous, can't resolve repeats
- Large k (e.g., k=10): very specific edges, but harder to connect at repeat boundaries; mismatches break the graph
- Practical choice: k=31–127 for Illumina; SPAdes uses multiple k values simultaneously

**What complicates real graphs** (from lecture):
- Sequencing errors → spurious k-mers → "tips" and erroneous edges
- Diploid/polyploid organisms → heterozygous SNPs create "bubbles" in the graph
- Repeats → multiple paths through the same node ("complex nodes")

In [ ]:
def build_debruijn_graph(reads: list[str], k: int) -> dict[str, list[str]]:
    """Build a de Bruijn graph from reads.
    
    Nodes are (k-1)-mers. Edges are k-mers.
    Returns adjacency list: {left_kmer: [right_kmer, ...]}
    """
    graph = defaultdict(list)
    for read in reads:
        for i in range(len(read) - k + 1):
            kmer = read[i:i + k]
            left = kmer[:-1]   # (k-1)-mer prefix
            right = kmer[1:]   # (k-1)-mer suffix
            graph[left].append(right)
    return dict(graph)


def get_kmers(sequence: str, k: int) -> list[str]:
    """Extract all k-mers from a sequence."""
    return [sequence[i:i + k] for i in range(len(sequence) - k + 1)]


def eulerian_path(graph: dict[str, list[str]]) -> list[str] | None:
    """Find an Eulerian path in a directed graph using Hierholzer's algorithm."""
    # Make a mutable copy
    adj = {node: list(edges) for node, edges in graph.items()}
    # Count in-degree and out-degree
    out_deg = {node: len(edges) for node, edges in adj.items()}
    in_deg = defaultdict(int)
    for node, edges in adj.items():
        for dest in edges:
            in_deg[dest] += 1
    
    all_nodes = set(out_deg) | set(in_deg)
    
    # Find start node: out_deg - in_deg == 1 (or any node with edges if Eulerian circuit)
    start = None
    for node in all_nodes:
        diff = out_deg.get(node, 0) - in_deg.get(node, 0)
        if diff == 1:
            start = node
            break
    if start is None:
        start = next(iter(out_deg))
    
    # Hierholzer
    stack = [start]
    path = []
    while stack:
        v = stack[-1]
        if adj.get(v):
            stack.append(adj[v].pop())
        else:
            path.append(stack.pop())
    
    path.reverse()
    return path


def path_to_sequence(path: list[str]) -> str:
    """Reconstruct sequence from a de Bruijn path of (k-1)-mers."""
    if not path:
        return ""
    return path[0] + "".join(node[-1] for node in path[1:])


# Demonstrate k-mer decomposition (lecture slides 22-35)
read1 = "AGCTACAGTATGC"
read2 = "TATGCTTATCTGA"

for k in [5, 10]:
    kmers1 = get_kmers(read1, k)
    kmers2 = get_kmers(read2, k)
    shared = set(kmers1) & set(kmers2)
    print(f"k={k}: {len(kmers1)} k-mers from read1, {len(kmers2)} from read2, {len(shared)} shared")
    if shared:
        print(f"  Shared (junction): {shared}")
print()

# Build graph for k=5 (from lecture)
reads_toy = ["AGCTACAGTATGC", "TATGCTTATCTGA"]
g = build_debruijn_graph(reads_toy, k=5)
print("De Bruijn graph (k=5), edges (k-mer = node_left -> node_right):")
for src in sorted(g.keys()):
    for dst in g[src]:
        kmer = src + dst[-1]
        print(f"  {src} -> {dst}  [{kmer}]")

In [ ]:
# Fix the adjacency copy (use items() properly) and assemble a small sequence
def assemble_debruijn(reads: list[str], k: int) -> list[str]:
    """Assemble reads using de Bruijn graph + Eulerian path.
    Returns list of contigs (one per connected component with valid path).
    """
    graph = build_debruijn_graph(reads, k)
    
    # Count in/out degrees
    out_deg = defaultdict(int)
    in_deg = defaultdict(int)
    for src, dsts in graph.items():
        for dst in dsts:
            out_deg[src] += 1
            in_deg[dst] += 1

    all_nodes = set(out_deg) | set(in_deg)
    print(f"Graph: {len(all_nodes)} nodes, {sum(out_deg.values())} edges")

    # Show degree imbalance (indicates start/end of path)
    for node in sorted(all_nodes):
        o = out_deg.get(node, 0)
        i = in_deg.get(node, 0)
        if o != i:
            label = "START" if o > i else "END"
            print(f"  {node}: out={o} in={i} [{label}]")

    # Greedy path: follow unambiguous chains (simplified)
    adj = defaultdict(list)
    for src, dsts in graph.items():
        adj[src].extend(dsts)

    visited_edges = set()
    contigs = []

    # Find start nodes (out > in, or any unvisited)
    starts = [n for n in all_nodes if out_deg.get(n, 0) > in_deg.get(n, 0)]
    if not starts:
        starts = list(out_deg.keys())

    for start in starts:
        if not adj[start]:
            continue
        path = [start]
        current = start
        while adj[current]:
            nxt = adj[current].pop(0)
            edge_key = (current, nxt, len(path))
            path.append(nxt)
            current = nxt
        if len(path) > 1:
            contigs.append(path_to_sequence(path))

    return contigs


# Test with a known sequence fragmented into short reads
target = "AGCTACAGTATGCTTATCTGATAC"
read_len = 10
step = 3
test_reads = [target[i:i+read_len] for i in range(0, len(target) - read_len + 1, step)]

print(f"Target:    {target} (len={len(target)})")
print(f"Reads ({len(test_reads)} reads, length {read_len}, step {step}):")
for r in test_reads:
    print(f"  {r}")
print()

contigs = assemble_debruijn(test_reads, k=7)
print(f"\nAssembled contigs:")
for c in contigs:
    match = "MATCH" if c == target else ("PARTIAL" if c in target else "MISMATCH")
    print(f"  {c} [{match}]")

### 1.4 Real-World Assemblers

| Assembler | Read Type | Algorithm | Best For |
|-----------|-----------|-----------|----------|
| **SPAdes** | Illumina (short) | Multi-k de Bruijn | Bacteria, small eukaryotes, metagenomes |
| **Velvet** | Illumina (short) | De Bruijn | Early short-read assembler |
| **Canu** | PacBio CLR / ONT | OLC + correction | Large genomes, high repeat content |
| **Flye** | PacBio CLR / ONT / HiFi | Repeat graph | Fast, handles high error rates |
| **hifiasm** | PacBio HiFi | Graph-based | Haplotype-resolved assembly, state of the art |
| **verkko** | HiFi + ONT ultra-long | Graph-based | T2T-quality assemblies |

**SPAdes** improvements over naive de Bruijn:
1. Runs multiple k values (e.g., 21, 33, 55, 77) and merges the graphs
2. Uses paired-end information to resolve the graph
3. Error correction via k-mer count thresholds (low-count k-mers are errors)
4. Performs graph simplification: tip clipping, bubble popping, low-coverage edge removal

```bash
# SPAdes basic usage
spades.py -1 reads_R1.fastq.gz -2 reads_R2.fastq.gz -o spades_output/ -t 8

# For metagenomes
spades.py --meta -1 reads_R1.fastq.gz -2 reads_R2.fastq.gz -o metaspades_output/

# Flye for long reads
flye --nano-raw reads.fastq.gz --genome-size 5m --out-dir flye_output/ --threads 8

# hifiasm for PacBio HiFi
hifiasm -o assembly -t 16 hifi_reads.fastq.gz
```

---

## Part 2: Assembly Quality Assessment

### 2.1 N50 and Related Metrics

After assembly, you have a set of contigs (continuous sequences). Quality is measured by how large and complete they are:

**N50**: The length L such that contigs of length ≥ L together cover at least 50% of the total assembly length. A higher N50 means fewer, larger contigs.

**L50**: The minimum number of contigs needed to cover 50% of the total assembly length.

**NG50**: Like N50 but uses the *estimated genome size* as denominator instead of total assembly length. This penalizes assemblies that are shorter than the expected genome size.

In [ ]:
def calculate_n50_l50(contig_lengths: list[int]) -> tuple[int, int]:
    """Calculate N50 and L50 from a list of contig lengths."""
    sorted_lengths = sorted(contig_lengths, reverse=True)
    total = sum(sorted_lengths)
    threshold = total / 2
    cumsum = 0
    for i, length in enumerate(sorted_lengths):
        cumsum += length
        if cumsum >= threshold:
            return length, i + 1  # N50, L50
    return sorted_lengths[-1], len(sorted_lengths)


def calculate_ng50(contig_lengths: list[int], genome_size: int) -> int:
    """Calculate NG50: N50 using genome_size as the denominator."""
    sorted_lengths = sorted(contig_lengths, reverse=True)
    threshold = genome_size / 2
    cumsum = 0
    for length in sorted_lengths:
        cumsum += length
        if cumsum >= threshold:
            return length
    return 0  # Assembly shorter than genome estimate


def assembly_stats(contig_lengths: list[int], genome_size: int | None = None) -> dict:
    """Compute a summary of assembly statistics."""
    lengths = sorted(contig_lengths, reverse=True)
    n50, l50 = calculate_n50_l50(lengths)
    stats = {
        "num_contigs": len(lengths),
        "total_length": sum(lengths),
        "largest_contig": lengths[0],
        "N50": n50,
        "L50": l50,
        "N90": None,
        "NG50": None,
    }
    # N90
    threshold_90 = stats["total_length"] * 0.9
    cumsum = 0
    for length in lengths:
        cumsum += length
        if cumsum >= threshold_90:
            stats["N90"] = length
            break
    # NG50
    if genome_size:
        stats["NG50"] = calculate_ng50(lengths, genome_size)
    return stats


# Compare two hypothetical assemblies for a 5 Mb genome
genome_size = 5_000_000

# Assembly A: many small contigs (fragmented)
rng = np.random.default_rng(0)
assembly_a = sorted(rng.integers(500, 20_000, size=800).tolist(), reverse=True)

# Assembly B: fewer large contigs (more contiguous)
assembly_b = sorted(rng.integers(5_000, 200_000, size=80).tolist(), reverse=True)

for name, lengths in [("Assembly A (fragmented)", assembly_a),
                       ("Assembly B (contiguous)", assembly_b)]:
    s = assembly_stats(lengths, genome_size)
    print(f"{name}")
    for k, v in s.items():
        if v is not None:
            print(f"  {k:>16}: {v:>12,}")
    print()

In [ ]:
# Visualize N50 concept
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, (name, lengths) in zip(axes, [
    ("Assembly A (fragmented)", assembly_a),
    ("Assembly B (contiguous)", assembly_b)
]):
    sorted_l = sorted(lengths, reverse=True)
    total = sum(sorted_l)
    cumsum = np.cumsum(sorted_l)
    n50, l50 = calculate_n50_l50(sorted_l)

    ax.bar(range(len(sorted_l)), sorted_l, width=1.0, color="steelblue", alpha=0.7)
    ax.axvline(l50 - 1, color="red", linestyle="--", lw=1.5, label=f"L50={l50}")
    ax.axhline(n50, color="orange", linestyle="--", lw=1.5, label=f"N50={n50:,}")
    ax.set_xlabel("Contig rank (longest first)")
    ax.set_ylabel("Contig length (bp)")
    ax.set_title(name)
    ax.legend()

plt.suptitle("N50 / L50 visualization", fontsize=13)
plt.tight_layout()
plt.show()

### 2.2 BUSCO: Completeness via Single-Copy Orthologs

**BUSCO** (Benchmarking Universal Single-Copy Orthologs) answers: *does the assembly contain the expected set of conserved genes?*

- Uses curated databases of genes that occur once per genome in >90% of species in a lineage
- For each gene, reports: **Complete** (full-length, single-copy), **Duplicated** (multiple copies), **Fragmented** (partial), **Missing**

```bash
busco -i assembly.fasta -l bacteria_odb10 -o busco_output -m genome
```

**Interpreting BUSCO output**:

| Result | Example | Interpretation |
|--------|---------|----------------|
| C:98.5% S:97.1% D:1.4% F:0.8% M:0.7% | Excellent | Near-complete, low duplication |
| C:65% M:35% | Poor | Assembly missing one-third of expected genes |
| C:95% D:40% | Contamination? | High duplication suggests contamination or polyploidy |

### 2.3 QUAST Metrics

QUAST provides a comprehensive HTML report with:
- Total assembly length and number of contigs
- N50/N75/N90, L50/L75/L90
- If a reference is provided: misassemblies, mismatches, indels per 100 kb
- Genome fraction covered, duplication ratio

```bash
quast.py assembly.fasta -r reference.fasta -o quast_output/ --threads 4
```

In [ ]:
# Simulate QUAST-style report and visualize

assemblies = {
    "Short reads only\n(SPAdes)": {"contigs": 450, "total_bp": 4_850_000, "N50": 28_000,  "busco": 94.2},
    "Long reads only\n(Flye)": {"contigs": 12,  "total_bp": 4_990_000, "N50": 680_000, "busco": 97.8},
    "Hybrid\n(SPAdes+long)": {"contigs": 8,   "total_bp": 5_010_000, "N50": 890_000, "busco": 98.5},
}

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
names = list(assemblies.keys())
colors = ["steelblue", "darkorange", "forestgreen"]

metrics = ["N50", "contigs", "busco"]
ylabels = ["N50 (bp)", "Number of contigs", "BUSCO completeness (%)"]
titles  = ["N50", "Contig count (lower is better)", "BUSCO completeness"]

for ax, metric, ylabel, title in zip(axes, metrics, ylabels, titles):
    values = [assemblies[n][metric] for n in names]
    bars = ax.bar(range(len(names)), values, color=colors, alpha=0.85)
    ax.set_xticks(range(len(names)))
    ax.set_xticklabels(names, fontsize=8)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() * 1.01,
                f"{val:,}" if isinstance(val, int) else f"{val:.1f}%",
                ha="center", va="bottom", fontsize=8)

plt.suptitle("Assembly quality comparison (simulated data)", fontsize=12)
plt.tight_layout()
plt.show()

print("Key observation: hybrid assembly achieves best N50 and BUSCO while minimizing contig count.")

---

## Part 3: Scaffolding and Gap Filling

### 3.1 From Contigs to Scaffolds

Contigs end at:
1. True chromosome ends (telomeres for linear molecules)
2. Coverage gaps (low-depth regions)
3. Branching points in the assembly graph — where a repeat causes ambiguity (most common)

**Scaffolding** uses long-range linking information to order and orient contigs, inserting `N` placeholders for gaps:

```
Contig A  [====AAAA====]  gap  [====BBBB====]  Contig B
                          NNN
Scaffold: AAAA...AAAA-NNN..NNN-BBBB...BBBB
```

### 3.2 Scaffolding Technologies

| Technology | Insert size | Resolution | Use |
|-----------|-------------|-----------|-----|
| **Paired-end** | 300–500 bp | Closes small gaps | Basic, always available |
| **Mate-pair** | 2–10 kb | Links contigs across repeats | Standard for WGS |
| **Long reads (ONT/HiFi)** | 10 kb – 2 Mb | Spans most repeats | Modern standard |
| **Hi-C** | Chromosome-scale | Chromosome-level ordering | T2T assemblies |
| **Optical mapping** | 100 kb–1 Mb | Validation and ordering | 10x Genomics Chromium |

**Paired-end / mate-pair scaffolding logic** (from lecture slide 54):
- We know the insert size (mean ± 10–20%)
- If both reads of a pair map to different contigs with consistent orientation and distance → the contigs are linked
- Many such links are combined to build a **scaffold graph**

**Hi-C scaffolding**:
- Chromatin is cross-linked in 3D, so nearby genomic regions become linked
- Hi-C contact frequency ~ 1/distance → provides both ordering and relative distance
- Tools: 3D-DNA, SALSA2, YaHS
- Enables chromosome-level ("chromosome-scale") assemblies

### 3.3 Gap Filling

After scaffolding, gaps (N regions) remain. Long reads that span the gap can fill them:
- Map long reads to scaffold ends flanking the gap
- Reads that map to both ends span the gap → extract their sequence
- Tools: LR_Gapcloser, Sealer, TGS-GapCloser

### 3.4 Understanding the Assembly Graph

Modern assemblers output a **GFA** (Graphical Fragment Assembly) file representing the assembly graph:
- **S** lines: sequences (contigs/unitigs)
- **L** lines: links (edges) between sequences with overlap info
- Tools like **Bandage** visualize GFA files
- Circular contigs (no dead ends) in GFA → likely complete circular chromosomes or plasmids

```bash
# View assembly graph
bandage image assembly_graph.gfa graph.png

# Gap filling with long reads
TGS-GapCloser --scaff scaffolds.fasta --reads long_reads.fastq --output gap_filled --thread 8
```

---

## Part 4: Read Mapping Algorithms in Depth

### 4.1 The Burrows-Wheeler Transform (BWT)

Reference mapping (covered in 3.01 as "what BWA does") is powered under the hood by the Burrows-Wheeler Transform — an invertible string permutation that enables extremely efficient substring search.

**BWT construction**:
1. Append a terminal character `$` (lexicographically smallest)
2. Form all rotations of the string
3. Sort the rotations lexicographically
4. The last column of the sorted matrix is the BWT

**Key property**: the BWT clusters similar characters together, making it highly compressible and enabling pattern search via the **FM-index**.

In [ ]:
def build_bwt(text: str) -> tuple[str, list[str]]:
    """Build Burrows-Wheeler Transform of text.
    
    Returns (bwt_string, sorted_rotations).
    """
    if not text.endswith("$"):
        text = text + "$"
    n = len(text)
    rotations = [text[i:] + text[:i] for i in range(n)]
    sorted_rotations = sorted(rotations)
    bwt = "".join(r[-1] for r in sorted_rotations)
    return bwt, sorted_rotations


def invert_bwt(bwt: str) -> str:
    """Recover original string from BWT using LF-mapping."""
    n = len(bwt)
    # Build F (first column) = sorted BWT
    f = sorted(bwt)
    # LF mapping: for each char in BWT, map to its rank in F
    # Count occurrences before position i in BWT
    lf = [0] * n
    char_count_bwt = defaultdict(int)
    char_count_f = defaultdict(int)
    char_first_pos = {}  # first occurrence of each char in F
    for i, c in enumerate(f):
        if c not in char_first_pos:
            char_first_pos[c] = i

    rank_in_bwt = defaultdict(int)
    bwt_rank = []
    for c in bwt:
        bwt_rank.append(rank_in_bwt[c])
        rank_in_bwt[c] += 1

    result = []
    row = 0  # start at the row with '$' in F
    for _ in range(n - 1):
        c = bwt[row]
        result.append(c)
        row = char_first_pos[c] + bwt_rank[row]

    return "".join(reversed(result))


# Demonstrate BWT construction
text = "BANANA"
bwt, sorted_rots = build_bwt(text)

print(f"Original: {text}")
print(f"BWT:      {bwt}")
print()
print(f"{'i':>3}  {'Rotation':>12}  {'Last char':>10}")
print("-" * 30)
for i, rot in enumerate(sorted_rots):
    print(f"{i:>3}  {rot:>12}  {rot[-1]:>10}")

# Verify invertibility
recovered = invert_bwt(bwt)
print(f"\nRecovered: {recovered}  (matches original: {recovered == text})")

# Show compression property
for seq in ["ACGTACGTACGT", "AAAAACCCCCGGGGG", "GATTACAGATTAC"]:
    b, _ = build_bwt(seq)
    # Run-length compress
    runs = sum(1 for i in range(len(b)) if i == 0 or b[i] != b[i-1])
    print(f"{seq:>20} -> BWT: {b:>16} (runs: {runs}/{len(b)})")

### 4.2 FM-Index and Backward Search

The FM-index (Full-text Minute-space index) adds two data structures to the BWT that enable O(m) pattern search for a pattern of length m:

- **C[c]**: for each character c, the number of characters lexicographically smaller than c in the text
- **Occ[c, i]**: the number of occurrences of character c in BWT[1..i]

**Backward search** (LF-mapping): match the pattern right-to-left. At each step, maintain a range [lo, hi] of sorted rows in the BWT matrix that match the current suffix of the pattern:

```
lo = C[pattern[i]] + Occ[pattern[i], lo - 1] + 1
hi = C[pattern[i]] + Occ[pattern[i], hi]
```

If lo > hi at any point, the pattern is not present. Otherwise, [lo, hi] gives all match positions.

### 4.3 How BWA-MEM Works

BWA-MEM (used in 3.01) builds on FM-index with several additions:

1. **Seeding**: Find exact-match seeds (SMEMs — Super-Maximal Exact Matches) between read and reference using FM-index backward search
2. **Seed chaining**: Group seeds that are collinear (same relative order in read and reference) into candidate alignment chains
3. **Extension**: For the best chain(s), extend using Smith-Waterman (banded) alignment to handle mismatches and indels
4. **Output**: SAM record with CIGAR string, mapping quality, and optional secondary alignments

**Mapping quality** (MAPQ) represents confidence that the read maps to the reported location. MAPQ = −10 log₁₀(P(wrong mapping)):
- MAPQ 0: read maps equally well to multiple locations (multimapper)
- MAPQ 20: 1% chance of wrong mapping
- MAPQ 60: the maximum reported by BWA-MEM (practical upper bound)

**Connection to Tier 4**: The suffix array (used in many aligners) and BWT are deeply related — the BWT is equivalent to the suffix array. The theoretical foundation is covered in Tier 4 String Algorithms.

In [ ]:
def fm_backward_search(bwt: str, pattern: str) -> tuple[int, int]:
    """FM-index backward search.
    
    Returns (lo, hi) interval in sorted suffix array.
    If lo > hi, pattern not found.
    """
    n = len(bwt)
    f = sorted(bwt)  # First column

    # C[c] = number of chars lexicographically < c
    chars = sorted(set(bwt))
    c_table = {}
    count = 0
    for ch in chars:
        c_table[ch] = count
        count += f.count(ch)

    # Occ[c, i] = count of c in bwt[0..i-1]
    def occ(c, i):
        return bwt[:i].count(c)

    lo, hi = 0, n - 1
    for ch in reversed(pattern):
        if ch not in c_table:
            return 1, 0  # not found
        lo = c_table[ch] + occ(ch, lo)
        hi = c_table[ch] + occ(ch, hi + 1) - 1
        if lo > hi:
            return 1, 0  # not found

    return lo, hi


# Demonstrate search
reference = "ACGTACGTATGCACGT"
bwt_ref, sorted_rots = build_bwt(reference)

print(f"Reference: {reference}")
print(f"BWT:       {bwt_ref}")
print()

patterns = ["ACGT", "TATGC", "GGGGG", "ACG"]
print(f"{'Pattern':>10} {'lo':>5} {'hi':>5} {'count':>7} {'found':>7}")
print("-" * 40)
for pat in patterns:
    lo, hi = fm_backward_search(bwt_ref, pat)
    found = lo <= hi
    count = hi - lo + 1 if found else 0
    # Verify with naive search
    naive_count = reference.count(pat)
    match_str = "OK" if count == naive_count else f"ERROR(naive={naive_count})"
    print(f"{pat:>10} {lo:>5} {hi:>5} {count:>7} {str(found):>7}  [{match_str}]")

---

## Part 5: Advanced Quality Control

### 5.1 MultiQC: Aggregated QC Reports

When processing many samples, running FastQC on each gives separate HTML reports — MultiQC aggregates them:

```bash
# Run FastQC on all samples
fastqc *.fastq.gz -o fastqc_results/ -t 8

# Aggregate with MultiQC
multiqc fastqc_results/ -o multiqc_report/
```

MultiQC produces a single HTML report with heatmaps for pass/fail per sample, and overlaid plots for per-base quality, GC content, duplication, etc. It also supports many other tools (STAR, HISAT2, samtools stats, GATK, BUSCO, QUAST).

### 5.2 K-mer Based QC: GenomeScope and Genome Size Estimation

Before assembly, k-mer frequency histograms from raw reads reveal:
- **Genome size**: total unique k-mers / (mean coverage)
- **Heterozygosity rate**: diploid genomes show a bimodal distribution
- **Ploidy**: polyploid genomes show multi-modal distributions
- **Repeat content**: the "shoulder" at high k-mer frequencies

In [ ]:
def simulate_kmer_histogram(genome_size: int, coverage: float,
                             error_rate: float = 0.01,
                             heterozygosity: float = 0.01) -> dict[int, int]:
    """Simulate a k-mer frequency histogram for GenomeScope-style analysis.
    
    In a diploid genome with heterozygosity h:
    - Heterozygous positions produce k-mers at ~cov/2 depth
    - Homozygous positions produce k-mers at ~cov depth
    - Errors produce k-mers at depth 1-2
    """
    from scipy.stats import poisson
    hist = defaultdict(int)

    # Error k-mers: Poisson with very low lambda
    error_kmers = int(genome_size * error_rate * 20)
    for _ in range(error_kmers):
        count = max(1, int(np.random.poisson(1.5)))
        hist[count] += 1

    # Heterozygous k-mers (half coverage peak)
    het_kmers = int(genome_size * heterozygosity)
    for _ in range(het_kmers):
        count = max(1, int(np.random.poisson(coverage / 2)))
        hist[count] += 1

    # Homozygous k-mers (full coverage peak)
    hom_kmers = genome_size - het_kmers
    for _ in range(min(hom_kmers, 50_000)):  # subsample for speed
        count = max(1, int(np.random.poisson(coverage)))
        hist[count] += 1

    return dict(hist)


fig, axes = plt.subplots(1, 2, figsize=(13, 4))

params = [
    ("Haploid (bacterium)",    1_000_000, 60, 0.0,  0.005),
    ("Diploid (eukaryote, 1% het)", 5_000_000, 40, 0.01, 0.01),
]

for ax, (title, gs, cov, het, err) in zip(axes, params):
    np.random.seed(7)
    hist = simulate_kmer_histogram(gs, cov, err, het)
    xs = sorted(k for k in hist if 1 <= k <= cov * 3)
    ys = [hist[x] for x in xs]
    ax.bar(xs, ys, width=1, color="steelblue", alpha=0.8)
    ax.axvline(cov // 2, color="orange", linestyle="--", lw=1.5, label=f"het peak (~{cov//2}x)")
    ax.axvline(cov, color="red", linestyle="--", lw=1.5, label=f"hom peak (~{cov}x)")
    ax.set_xlabel("k-mer depth")
    ax.set_ylabel("Number of distinct k-mers")
    ax.set_title(title)
    ax.set_xlim(0, cov * 2.5)
    ax.legend(fontsize=8)

plt.suptitle("K-mer frequency histograms (simulated)", fontsize=12)
plt.tight_layout()
plt.show()

print("GenomeScope fits a negative binomial model to this histogram to estimate:")
print("  - Genome size (from total k-mers / mean coverage)")
print("  - Heterozygosity (from ratio of het-peak to hom-peak areas)")
print("  - Repeat content (from excess k-mers above the main peak)")

### 5.3 Contamination Detection

Assembly contamination (reads from bacteria, fungi, human) is a major problem, especially for environmental samples.

**Approaches**:

| Tool | Method | Speed | Use case |
|------|--------|-------|----------|
| **Kraken2** | k-mer matching to reference DB | Very fast (minutes) | Read-level classification |
| **BLAST** | Sequence alignment | Slow | Contig-level validation |
| **blobtools** | Coverage + GC + taxonomy | Medium | Visualize and filter contigs |
| **FCS-GX** (NCBI) | BLAST-based | Fast | Submission-ready decontamination |

**Blob plots** (blobtools): scatter plot of GC content vs. coverage, coloured by taxonomic assignment. Contaminant contigs cluster away from the main species cloud.

### 5.4 Coverage Analysis and Uniformity

Even after QC filtering, non-uniform coverage reveals assembly problems:
- Very low coverage regions: possible assembly errors, misjoins
- Very high coverage regions: repetitive elements collapsed into a single contig
- Expected coverage for a contig: total_reads × read_length / genome_size

In [ ]:
# Simulate coverage depth profile across contigs
np.random.seed(1)

genome_size = 5_000_000
expected_cov = 50
positions = np.arange(0, genome_size, 1000)

# Base coverage: Poisson around expected
coverage = np.random.poisson(expected_cov, size=len(positions)).astype(float)

# Add features
# Low-coverage gap (possible assembly problem)
coverage[500:520] = np.random.poisson(5, size=20)
# High-coverage repeat
coverage[1200:1240] = np.random.poisson(expected_cov * 4, size=40)
# Telomeric region (very low coverage)
coverage[:30] = np.random.poisson(3, size=30)
coverage[-30:] = np.random.poisson(3, size=30)

fig, ax = plt.subplots(figsize=(13, 3.5))
ax.fill_between(positions / 1e6, coverage, alpha=0.6, color="steelblue")
ax.axhline(expected_cov, color="red", linestyle="--", lw=1, label=f"Expected ({expected_cov}x)")
ax.axhline(expected_cov * 0.3, color="orange", linestyle="--", lw=1, label="Low coverage threshold (0.3x)")
ax.axhline(expected_cov * 2.5, color="purple", linestyle="--", lw=1, label="High coverage (repeat signal, 2.5x)")

# Annotate features
ax.annotate("Coverage gap\n(misassembly?)", xy=(0.5, 8), xytext=(0.65, 60),
            arrowprops=dict(arrowstyle="->", color="black"), fontsize=8)
ax.annotate("Repeat collapse", xy=(1.22, 200), xytext=(1.4, 200),
            arrowprops=dict(arrowstyle="->", color="black"), fontsize=8)

ax.set_xlabel("Genomic position (Mb)")
ax.set_ylabel("Read depth")
ax.set_title("Coverage depth profile (simulated)")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

print(f"Mean coverage: {coverage.mean():.1f}x")
print(f"Coverage CoV (lower is more uniform): {coverage.std()/coverage.mean():.3f}")
pct_low = (coverage < expected_cov * 0.3).mean() * 100
pct_high = (coverage > expected_cov * 2.5).mean() * 100
print(f"Fraction below 0.3x threshold: {pct_low:.1f}%")
print(f"Fraction above 2.5x threshold (repeats): {pct_high:.1f}%")

---

## Part 6: Long-Read Sequencing Deep Dive

### 6.1 Oxford Nanopore Technology (ONT)

**Principle**: DNA passes through a protein nanopore embedded in a membrane. As each k-mer of bases occupies the constriction zone it causes a characteristic disruption to the ionic current; the resulting time-series ("squiggle") is decoded by a neural network (basecaller) into nucleotide sequence.

**Basecalling with Dorado** (the current GPU-accelerated basecaller, replacing Guppy):
- `fast` model: ~95–97% per-read accuracy, highest throughput
- `hac` (high-accuracy): ~97–99% per-read accuracy — standard choice
- `sup` (super-accuracy): ~99%+ per-read, 5–10× slower than `hac`

Raw signal is stored in **POD5** format (columnar Arrow-based binary, replacing legacy HDF5-based FAST5). Output from Dorado is an unaligned BAM preserving all per-read metadata.

**Error profile**: historically dominated by homopolymer indels (R9.4.1); the R10.4.1 dual-pore chemistry uses a 9-mer sensing window and dramatically reduces homopolymer errors.

**Adaptive sampling (ReadFish/MinKNOW)**: after reading ~400 bp the pore can apply reverse voltage to eject an unwanted molecule in real time, enabling target enrichment or host depletion without physical capture.

**Read lengths**: typical N50 of 15–25 kb (R9.4.1) or 20–40 kb (R10.4.1); ultra-long (UL) extraction protocols yield reads up to 4 Mb.

### 6.2 PacBio HiFi (CCS reads)

**Principle**: A single circular DNA molecule is sequenced repeatedly by the same polymerase (SMRT sequencing). Multiple passes around the same molecule are merged into a **Circular Consensus Sequence (CCS)**, cancelling random errors.

**HiFi accuracy**: ≥Q30 (99.9%) per-read accuracy from ≥3 passes; typical library reads pass ≥8 times. HiFi assemblies (hifiasm) require no polishing.

| Property | PacBio CLR | PacBio HiFi (Revio) | ONT R9.4.1 | ONT R10.4.1 |
|----------|-----------|---------------------|-----------|------------|
| Read length | Up to 100 kb | 10–25 kb (CCS) | 50 bp – 4 Mb | 50 bp – 4 Mb |
| Per-read accuracy | ~85–90% | ≥99.9% (Q30) | ~95% | ~97–99% |
| Consensus accuracy | N/A | ≥99.9% | 99.5%+ | 99.9%+ |
| Throughput | 15 Gb/SMRT cell | ~90 Gb (Revio) | 30–50 Gb | 50–120 Gb |
| Error type | Random | Very rare | Homopolymer indels | Reduced homopolymer errors |
| Best use | Legacy large genomes | High-accuracy assembly/phasing | Ultra-long scaffolding | Balanced accuracy + length |

### 6.3 Hybrid Assembly Strategies

Combining technologies leverages each platform's strengths:

In [ ]:
# Technology comparison radar chart
from matplotlib.patches import FancyArrowPatch

categories = ["Read length", "Raw accuracy", "Throughput", "Cost/Gb", "Homopolymer\naccuracy"]
# Scale 0-10 (10=best)
platforms = {
    "Illumina (short)": [1, 10, 9, 10, 9],
    "PacBio HiFi":      [6, 10, 7, 4,  9],
    "ONT R10.4":        [10, 8, 8, 7,  7],
    "ONT UL":           [10, 7, 5, 6,  6],
}

N = len(categories)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(7, 6), subplot_kw=dict(polar=True))
colors_map = ["steelblue", "darkorange", "forestgreen", "firebrick"]

for (name, values), color in zip(platforms.items(), colors_map):
    vals = values + values[:1]
    ax.plot(angles, vals, "o-", linewidth=2, label=name, color=color)
    ax.fill(angles, vals, alpha=0.08, color=color)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, size=9)
ax.set_yticks([2, 4, 6, 8, 10])
ax.set_yticklabels(["2", "4", "6", "8", "10"], size=7)
ax.set_ylim(0, 10)
ax.set_title("Sequencing platform comparison", pad=15)
ax.legend(loc="upper right", bbox_to_anchor=(1.35, 1.1), fontsize=9)
plt.tight_layout()
plt.show()

# Hybrid assembly decision guide
print("\nHybrid assembly strategies:")
strategies = [
    ("Bacteria/virus (5 Mb)",    "ONT or HiFi alone",             "Complete circular chromosomes"),
    ("Small eukaryote (50 Mb)",  "HiFi + Hi-C",                   "Chromosome-scale, phased"),
    ("Large plant (3 Gb+)",      "HiFi + ONT UL + Hi-C",          "T2T or near-T2T"),
    ("Human clinical WGS",       "Illumina 30x",                  "Cost-effective variant calling"),
    ("Human SV discovery",       "ONT 30x or HiFi 30x",           "Structural variant resolution"),
    ("Metagenome assembly",      "Illumina + ONT long",           "Binning + contig linking"),
]
print(f"{'Use case':>25}  {'Strategy':>30}  {'Goal'}")
print("-" * 80)
for uc, strat, goal in strategies:
    print(f"{uc:>25}  {strat:>30}  {goal}")

### 6.4 Polishing: Correcting Long-Read Assemblies

Long-read assemblies (especially CLR/ONT) contain systematic errors that need correction:

**Polishing workflow**:
1. Align short reads (Illumina) or HiFi reads back to the draft assembly
2. Call consensus at each position using the aligned reads
3. Correct errors in the draft

**Tools**:
- **Medaka** (ONT): neural-network polisher for ONT assemblies using ONT reads
- **Pilon**: polishes with Illumina short reads
- **Racon**: fast, read-overlap-based polishing
- **NextPolish**: fast multi-round polisher

**Note**: HiFi assemblies (hifiasm output) typically require no polishing — read accuracy is already ≥Q30.

```bash
# Medaka polishing (ONT assembly)
medaka_consensus -i reads.fastq -d draft_assembly.fasta -o medaka_output/ -t 8

# Pilon polishing (with Illumina short reads)
bwa mem assembly.fasta short_R1.fastq short_R2.fastq | samtools sort > illumina.bam
samtools index illumina.bam
pilon --genome assembly.fasta --frags illumina.bam --output polished
```

---

## Exercises

**Exercise 1 — De Bruijn assembly:**
The following 5 reads were generated from an unknown 20 bp sequence:
```
ATCGATCG
TCGATCGA
CGATCGAT
GATCGATA
ATCGATAT
```
Build a de Bruijn graph with k=5. Draw the nodes and edges. What sequence do you assemble? Does a unique Eulerian path exist? Why or why not?

**Exercise 2 — N50 calculation:**
An assembly of a 4.5 Mb genome produces contigs of lengths (bp):
`[1200000, 950000, 620000, 450000, 380000, 220000, 180000, 95000, 45000, 12000]`

Calculate by hand (or code): N50, L50, NG50 (using genome size = 4.5 Mb). Is there a difference between N50 and NG50? Why?

**Exercise 3 — BWT and search:**
Construct the BWT of `MISSISSIPPI`. Use FM backward search to find how many times `SS` occurs. Verify by counting in the original string.

**Exercise 4 — Technology selection:**
You are given $50,000 USD to sequence and assemble the genome of a newly discovered fern species (estimated genome size: 12 Gb, high repeat content, diploid). Current sequencing costs approximately:
- Illumina 150 bp PE: $8 per Gb
- PacBio HiFi: $25 per Gb
- ONT PromethION: $10 per Gb
- Hi-C library: $800 per sample

Propose a sequencing strategy. What coverage would you target for each platform? What assembler(s) would you use? What do you expect the final assembly to look like (estimated N50, BUSCO)?

**Exercise 5 — Repeat analysis:**
You assemble a 200 Mb genome and get an assembly of only 120 Mb with 4,500 contigs. BUSCO completeness is 88%. GenomeScope estimates 40% repeat content. 
(a) What is likely causing the fragmented assembly? 
(b) What additional data would you collect to improve it? 
(c) Why might the assembly be 120 Mb rather than 200 Mb?

In [ ]:
# Exercise 2 solution scaffold
contigs_ex2 = [1_200_000, 950_000, 620_000, 450_000, 380_000,
               220_000, 180_000, 95_000, 45_000, 12_000]
genome_size_ex2 = 4_500_000

n50, l50 = calculate_n50_l50(contigs_ex2)
ng50 = calculate_ng50(contigs_ex2, genome_size_ex2)
total_asm = sum(contigs_ex2)

print(f"Total assembly length : {total_asm:>12,} bp")
print(f"Estimated genome size : {genome_size_ex2:>12,} bp")
print(f"Assembly / genome     : {total_asm/genome_size_ex2:.2%}")
print(f"N50                   : {n50:>12,} bp")
print(f"L50                   : {l50:>12} contigs")
print(f"NG50                  : {ng50:>12,} bp")
print()
print(f"N50 > NG50: {n50 > ng50}")
print("Explanation: N50 uses total assembly length as denominator.")
print("Since total assembly ({:,}) > genome size ({:,}), the N50 threshold".format(total_asm, genome_size_ex2))
print("is lower, making N50 >= NG50.")

In [ ]:
# Exercise 3 solution scaffold
mississippi_bwt, ms_sorted = build_bwt("MISSISSIPPI")
lo, hi = fm_backward_search(mississippi_bwt, "SS")
fm_count = hi - lo + 1 if lo <= hi else 0
naive_count = "MISSISSIPPI".count("SS")

print(f"BWT of MISSISSIPPI: {mississippi_bwt}")
print(f"FM-index search for 'SS': lo={lo}, hi={hi}, count={fm_count}")
print(f"Naive count of 'SS' in 'MISSISSIPPI': {naive_count}")
print(f"Match: {fm_count == naive_count}")

print("\nSorted rotations (BWT matrix):")
for i, rot in enumerate(ms_sorted):
    marker = "<-- match" if lo <= i <= hi else ""
    print(f"  {i:>3}: {rot}  {marker}")

---

[< Previous: Numerical Methods for Bioinformatics](../16_Numerical_Methods_for_Bioinformatics/01_numerical_methods_for_bioinformatics.ipynb) | [Tier 3: Applied Bioinformatics Overview](../README.md) | [Next: Genome Assembly + Advanced Ngs: Overview >](01_genome_assembly_and_advanced_ngs.ipynb)

---